# Mission 3.2 — Housing Price Regression Arena
Run all cells in Google Colab, experiment with Ridge / RandomForestRegressor / GradientBoostingRegressor, and download `predictions.csv` to submit!


In [ ]:
import io
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

TRAIN_CSV = """id,area_norm,rooms,age_norm,distance_km,price_index
0,0.4728,4,0.1769,3.292,4.7726
1,0.6216,1,-0.3603,10.35,3.3746
2,-1.0197,3,-0.7624,5.52,1.8576
3,1.1786,1,0.3106,13.399,3.818
4,1.6781,2,-0.5244,8.782,5.707
5,-0.0786,5,0.3601,6.893,3.49
6,-0.6661,2,0.3524,10.805,1.1128
7,1.5165,5,1.1768,1.918,6.4681
8,1.4743,4,-0.5003,7.042,6.4569
9,1.0422,3,1.4213,1.551,4.5117
10,0.207,4,0.9761,9.474,3.1005
11,1.6288,1,-0.8643,11.534,5.433
12,-0.6091,2,1.3701,9.802,0.6613
13,1.9078,2,0.0694,12.841,5.9299
14,0.502,5,-0.1699,14.293,4.243
15,1.2589,1,0.5302,11.978,4.0677
16,-1.1906,4,-0.1583,6.871,1.6524
17,0.8487,4,-0.4021,2.89,5.8083
18,1.5461,3,1.4066,13.358,4.6231
19,-1.0974,3,0.3047,7.709,0.9657
20,0.4999,2,-0.7617,13.046,3.8181
21,-0.4821,3,0.4617,3.526,2.3243
22,-0.0365,1,1.0382,1.911,2.2315
23,-0.0696,3,0.7773,14.996,1.9487
24,0.5304,4,1.3383,7.116,3.646
25,-0.6676,5,1.2574,8.989,1.9598
26,1.7079,2,1.0779,2.143,5.8767
27,1.6815,3,-0.8338,3.488,7.0879
28,1.8116,2,0.9102,3.516,5.4339
29,-1.1529,1,0.5621,1.112,0.4898
30,-0.1583,2,-0.0655,4.415,3.0042
31,0.3878,1,0.9838,8.125,2.4272
32,0.6593,4,-0.7117,12.221,4.6761
33,-0.4174,4,-0.3544,7.853,2.8594
34,-0.4078,2,-0.0335,5.912,1.9281
35,1.3295,5,0.3885,9.039,6.1276
36,0.9939,5,0.7525,1.735,5.8449
37,0.5531,2,0.588,12.073,3.105
38,-0.9356,5,0.1696,3.695,2.1748
39,0.4202,2,1.1558,3.655,3.0335
40,1.6359,4,0.3781,14.459,5.7108
41,0.985,4,-0.2422,12.578,5.0586
42,0.6717,3,0.7051,9.922,3.856
43,-0.8053,5,-0.9102,11.966,2.576
44,1.998,2,0.7489,12.191,5.4448
45,1.0275,3,-0.1477,10.262,4.831
46,-1.1139,1,1.4617,8.705,-0.794
47,0.8114,3,1.1884,7.665,3.6473
48,1.2835,1,0.7923,6.148,4.0577
49,-0.4766,2,0.5963,10.814,1.2535
50,-0.1599,5,-0.6095,14.582,3.1871
51,-0.5882,2,0.7906,3.881,1.235
52,-0.6989,4,0.0155,1.922,2.7094
53,0.5492,3,-0.5344,6.939,4.4852
54,1.52,5,-0.5245,4.766,7.1682
55,1.0879,4,0.1127,7.645,5.4054
56,-0.4622,1,0.468,9.253,0.8599
57,0.1168,2,-0.706,5.443,3.5859
58,1.7358,4,0.3555,2.039,6.8342
59,1.7697,4,0.3182,1.579,7.0154
60,-0.1609,5,0.5889,14.514,2.6666
61,1.4573,4,0.2538,5.874,6.4047
62,-0.9941,5,-0.7656,2.07,2.9588
63,0.1806,2,-0.5728,13.126,2.8809
64,1.9969,3,0.0936,14.629,6.2686
65,1.0139,1,-0.0074,2.868,4.4334
66,-0.3236,5,-0.8589,8.318,3.7024
67,0.2921,5,-0.4224,4.006,4.804
68,-1.0031,3,0.7753,9.118,0.6644
69,-0.5992,4,-0.0011,2.993,2.9663
70,1.7882,2,0.2954,13.77,5.3181
71,0.3223,1,-0.3907,8.553,2.8651
72,1.705,3,1.0776,13.302,5.1799
73,1.1713,4,-0.3321,4.341,6.263
74,1.0393,1,0.3178,6.276,3.8916
75,1.5753,1,0.0487,14.752,4.7448
76,-1.1581,1,-0.8929,4.092,0.8055
77,-1.1797,2,-0.558,12.159,0.4048
78,-0.4872,3,0.6146,6.523,1.6914
79,-0.1979,5,-0.9427,3.076,4.4304
80,-0.0902,5,-0.8478,4.815,4.3818
81,0.2048,2,1.0865,3.802,2.7998
82,-0.8445,4,-0.0404,14.671,1.2145
83,-0.4421,2,-0.3491,13.189,1.3375
84,-0.2299,4,-0.5212,5.729,3.4201
85,-1.1273,5,0.1793,4.318,1.9639
86,-0.6643,3,-0.8845,3.186,2.5151
87,0.7128,4,0.6107,12.625,4.0511
88,-0.7103,4,0.1522,11.013,1.8024
89,1.8234,1,-0.5236,1.619,6.0925
90,-0.6935,5,0.2718,2.324,2.7712
91,0.4454,5,1.279,13.194,3.48
92,0.5207,3,1.2476,14.26,2.9059
93,-0.6359,2,-0.4042,3.03,2.0466
94,0.2579,1,0.5863,12.521,2.1104
95,-0.2398,5,0.1974,14.865,2.9077
96,0.5087,3,1.2225,1.303,3.8273
97,-0.4135,2,1.3806,7.033,1.1308
98,-0.9468,3,-0.4526,13.304,1.1608
99,-0.6154,1,0.3754,7.516,0.8612
100,-0.204,5,-0.3361,10.566,3.491
101,1.2191,4,0.7691,9.944,4.9677
102,1.1564,4,0.0684,7.626,5.6137
103,1.3289,3,1.1269,12.828,4.5461
104,0.481,1,-0.8821,11.432,3.2079
105,-0.4083,4,0.7542,2.282,2.7144
106,1.8156,2,1.4965,5.121,5.1532
107,0.8864,5,0.3443,6.937,5.3227
108,0.5954,4,0.3013,10.19,4.1357
109,-1.1713,5,0.2128,9.222,1.4015
110,1.9451,4,0.3654,3.403,7.0663
111,-0.6029,3,-0.228,7.936,2.1096
112,-0.3799,4,-0.0897,13.236,2.465
113,0.5228,3,-0.2155,3.953,4.4696
114,0.3232,2,0.5164,13.669,2.8159
115,-0.5424,5,-0.3864,9.853,2.6476
116,-0.167,4,0.0783,6.485,3.3496
117,0.2751,5,0.4424,12.848,3.8139
118,0.9612,1,-0.6224,5.028,4.644
119,-0.3762,4,0.6937,6.232,2.6029
120,-0.3538,5,1.4597,3.135,2.8632
121,-0.5457,5,0.0795,7.236,2.8521
122,-0.478,5,-0.9834,14.545,3.1254
123,-1.1501,4,1.3193,5.516,0.8296
124,-0.6481,3,-0.3182,10.01,2.176
125,1.6067,5,0.0558,2.068,7.191
126,0.3995,3,-0.6992,11.93,3.8308
127,-0.9527,1,-0.0802,5.507,0.7174
128,1.2104,5,-0.8409,2.705,7.0111
129,-0.5066,3,0.8822,11.876,1.0622
130,0.1591,5,1.1038,4.622,3.9104
131,0.1208,5,0.1973,13.64,3.3903
132,-0.0551,1,-0.3041,13.819,1.923
133,0.25,1,0.7675,7.115,2.2186
134,-1.0086,4,1.0723,2.325,1.2595
135,-1.0077,3,0.4595,5.827,1.1283
136,-0.9177,2,1.1709,1.054,1.0896
137,0.8541,1,0.0554,3.901,4.142
138,-0.5353,4,0.9283,11.632,1.8736
139,1.7346,1,0.3692,5.709,5.4546
140,0.8402,1,0.0012,11.779,3.5252
141,1.824,3,1.251,1.828,6.4276
142,-0.0129,3,-0.0588,7.644,2.8653
143,-0.7477,1,-0.9778,13.75,0.6969
144,0.3962,4,-0.3793,8.575,4.2847
145,1.3248,3,0.709,7.385,4.8727
146,1.525,2,0.3982,5.446,5.3999
147,0.3517,4,0.4071,10.234,3.7645
148,0.9285,3,-0.7193,12.411,4.8834
149,-0.3694,2,0.1236,7.312,2.0549
150,0.73,3,-0.2295,1.162,4.9376
151,-0.9481,2,1.488,12.856,-0.4239
152,1.7219,5,-0.4528,9.76,7.1497
153,0.6347,3,-0.8788,12.166,4.4286
154,0.6311,3,-0.4243,5.752,4.579
155,0.2835,1,0.0917,12.604,2.0479
156,1.6806,4,-0.2151,9.461,6.6073
157,-0.1244,5,1.1096,13.055,2.2745
158,-0.3185,4,0.7068,6.22,2.5869
159,0.6569,3,0.8487,8.154,3.6864
160,-0.6042,4,-0.0208,3.47,2.868
161,0.3986,4,0.5083,1.682,4.5834
162,-0.2232,1,1.2433,4.282,1.2438
163,0.4828,1,0.0592,3.953,3.258
164,-0.6282,4,-0.1398,5.666,2.511
165,1.49,4,0.5133,4.749,6.2894
166,1.5645,5,1.2009,13.156,5.7367
167,1.0882,3,-0.5443,7.747,5.4489
168,1.9952,3,0.4438,13.548,6.0777
169,-0.4711,5,-0.6537,1.533,3.7203
170,0.2113,5,0.1212,3.026,4.8079
171,1.8198,5,-0.7639,9.655,7.5981
172,0.2972,2,0.5885,6.299,3.1935
173,1.3945,2,0.9139,6.118,4.6833
174,1.2045,4,0.0845,12.362,5.3346
175,0.3737,5,0.8762,2.778,4.4369
176,0.5234,1,0.3689,6.324,3.0737
177,-0.171,1,0.7179,12.102,1.335
178,0.5733,5,1.4419,13.679,3.5395
179,-0.4151,1,0.4977,12.681,0.8825
180,0.2975,1,-0.4639,3.434,3.1809
181,-0.4283,2,0.8303,7.545,1.4904
182,-0.0655,5,-0.3257,14.529,3.4357
183,-0.2289,1,1.3557,11.526,0.686
184,-0.991,4,0.3216,8.289,1.5451
185,1.7146,2,-0.1031,2.546,6.3576
186,-0.661,2,-0.863,10.838,1.5878
187,1.5464,1,0.7788,13.702,4.098
188,-0.0633,5,-0.9914,14.801,3.7034
189,0.6719,1,0.9243,13.45,2.3415
190,0.6444,2,0.1852,5.575,3.7812
191,-1.1531,1,-0.1393,5.583,0.4604
192,1.0037,2,-0.2777,2.507,5.194
193,0.9053,1,1.4788,3.884,3.5068
194,-0.3384,5,0.7315,6.434,2.6888
195,1.8634,5,-0.2631,4.001,7.958
196,-1.1939,4,-0.7313,9.111,1.6174
197,1.8341,3,0.8092,12.982,5.808
198,-1.199,4,1.493,5.981,0.8067
199,1.3059,2,-0.7236,1.694,5.9809
200,1.7757,5,0.5163,5.754,6.813
201,0.5364,1,0.4436,3.649,3.17
202,1.672,5,-0.3033,12.59,6.5813
203,-1.1055,3,0.5228,1.042,1.2894
204,-0.6876,3,0.2892,9.251,1.4105
205,1.6746,5,-0.1157,7.278,7.0549
206,1.6799,3,1.3933,13.582,4.8252
207,1.5825,4,-0.1572,8.028,6.4861
208,1.5335,1,1.2903,13.075,4.1063
209,1.3249,1,0.2425,4.633,4.7123
210,-0.3171,4,1.0112,3.453,2.6084
211,-0.4582,1,0.8174,3.855,0.9799
212,0.2026,2,1.4593,12.41,1.9798
213,1.0004,5,-0.8688,5.413,6.2711
214,0.1158,1,0.6127,8.635,2.0054
215,0.2716,5,-0.4367,6.418,4.8911
216,1.1847,2,-0.6887,10.382,4.9005
217,-0.0248,3,-0.4171,13.017,2.7847
218,1.8558,5,-0.5029,2.659,7.6758
219,1.9574,5,0.5829,12.376,6.8368
220,0.616,2,0.082,13.302,3.5161
221,1.8943,1,0.739,9.563,4.9138
222,-0.67,5,1.3357,9.181,1.9313
223,0.0923,2,0.2218,3.061,3.1027
224,0.9854,1,0.322,2.975,4.3244
225,-0.1721,2,0.7391,7.637,1.94
226,-0.7733,2,1.4486,12.937,0.0292
227,-0.4256,2,0.0107,4.418,2.2169
228,1.4175,2,0.1972,13.166,4.6217
229,0.612,1,-0.4514,5.668,3.8686
230,-0.7201,1,-0.6862,4.153,1.6314
231,1.2277,2,0.5071,14.856,3.9756
232,1.1319,4,-0.0075,14.643,5.092
233,-0.2027,3,0.3367,5.558,2.5041
234,-0.7985,2,0.9778,12.184,0.1774
235,0.3312,4,-0.7933,14.003,4.0854
236,-1.0643,3,-0.6779,3.156,1.7657
237,1.7933,1,-0.0973,12.368,5.1114
238,1.9988,3,0.0644,7.979,6.6015
239,1.1992,1,-0.4355,13.488,4.1622"""
TEST_CSV = """id,area_norm,rooms,age_norm,distance_km
0,0.1815,1,0.8674,12.992
1,1.8013,2,-0.4481,5.217
2,1.4793,2,0.8002,7.98
3,0.6118,3,1.3387,9.156
4,-0.8618,3,1.444,5.632
5,0.2564,5,1.1097,5.437
6,1.5877,1,1.1919,2.844
7,1.004,4,0.4276,13.713
8,0.3657,4,-0.9283,9.467
9,0.2495,3,-0.4771,3.492
10,-0.9787,3,-0.7078,7.304
11,1.1463,3,-0.156,14.156
12,1.3636,2,0.7305,10.856
13,-0.7947,1,1.439,3.298
14,-0.6418,5,1.1906,2.351
15,-0.3392,5,1.0662,2.81
16,1.406,4,-0.5168,14.411
17,1.293,4,1.1274,12.043
18,0.2438,1,0.2421,8.815
19,-0.3118,4,-0.8243,10.8
20,0.8899,4,0.5946,9.423
21,0.1743,3,1.3952,8.469
22,0.0952,1,-0.0633,6.697
23,0.2056,2,-0.9012,2.327
24,0.5256,2,0.9666,2.373
25,1.0001,5,-0.091,9.319
26,-0.4583,5,0.1396,6.946
27,-0.9589,4,0.5689,9.98
28,1.0119,2,0.6308,10.352
29,1.8981,4,1.0049,2.74
30,-0.4097,4,-0.0175,8.894
31,0.2228,5,0.3583,13.568
32,1.8278,4,1.1954,2.393
33,-0.3253,4,0.1303,9.249
34,0.8399,5,-0.8896,13.824
35,-0.2025,4,-0.0351,13.087
36,-0.2067,1,0.3815,12.941
37,1.9368,1,-0.1266,10.258
38,1.9758,2,-0.7586,9.377
39,-1.0278,2,-0.8163,9.058
40,1.2831,3,0.3952,4.177
41,0.538,1,-0.1465,2.804
42,0.2579,4,1.2303,2.524
43,-1.0292,2,1.1234,12.015
44,-0.0715,1,0.4978,7.61
45,0.1696,3,0.5257,14.223
46,1.2528,2,1.2062,10.414
47,-0.6796,5,0.0598,5.927
48,1.9405,1,1.3355,1.782
49,1.6153,5,1.3804,3.286
50,0.4272,1,-0.0452,7.666
51,1.2046,2,0.2574,11.702
52,-0.7795,2,1.466,2.703
53,0.5759,2,-0.1126,9.6
54,-0.5107,2,-0.5316,11.718
55,0.3976,2,-0.9564,7.236
56,1.6173,5,0.9243,4.215
57,-0.0058,4,0.9491,5.571
58,0.6096,1,1.1416,2.248
59,0.9841,5,0.4329,13.996
60,1.8106,3,-0.0451,5.607
61,-0.0374,4,0.7959,10.49
62,-0.4999,5,0.9602,9.453
63,0.0774,2,0.5163,11.923
64,0.918,1,-0.9645,10.963
65,0.931,4,-0.9111,10.402
66,1.8453,1,-0.2476,13.502
67,-0.2565,3,-0.5971,14.545
68,-1.0379,4,0.7736,4.512
69,0.433,3,1.4361,13.955
70,0.2964,5,0.5861,9.338
71,0.7137,5,0.3162,1.589
72,-1.0702,1,-0.3262,3.165
73,1.2694,1,0.3575,1.266
74,-0.1006,3,1.3585,6.801
75,0.8545,4,-0.1606,11.65
76,1.355,2,-0.3025,11.394
77,-0.3192,3,-0.0411,2.266
78,1.1362,1,-0.5343,7.491
79,0.1488,5,1.1037,8.361"""

train = pd.read_csv(io.StringIO(TRAIN_CSV))
test_X = pd.read_csv(io.StringIO(TEST_CSV))
print("Train:", train.shape, "| Test:", test_X.shape)


In [ ]:
features = ["area_norm", "rooms", "age_norm", "distance_km"]
X_tr, y_tr = train[features], train["price_index"]

pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
cv_rmse = -cross_val_score(pipe, X_tr, y_tr, cv=5, scoring="neg_root_mean_squared_error").mean()
print(f"5-Fold CV RMSE: {cv_rmse:.4f}")

pipe.fit(X_tr, y_tr)
preds = pipe.predict(test_X[features])
pd.DataFrame({"id": test_X["id"], "prediction": preds}).to_csv("predictions.csv", index=False)
print("Generated predictions.csv! Download it from Colab Files sidebar.")
